# 02 — Mechanical Systems

**Topic**: SDOF oscillator, chain of oscillators, FE beam — matrix assembly, eigenfrequency convergence.

**Reference**: Krack & Gross (2019) §5; Euler-Bernoulli beam FEM

**Estimated runtime**: < 15 seconds

## Theory

All mechanical systems in NLvib share the linear equation of motion (K&G §2.1):

$$M\ddot{q} + D\dot{q} + Kq = f_{ext}(t) - f_{nl}(q, \dot{q}) \tag{1}$$

The matrices $M$ (mass), $D$ (damping), $K$ (stiffness) are assembled from element contributions and stored as `scipy.sparse.csr_matrix` for efficiency.

**Undamped natural frequencies** are eigenvalues of the generalised problem:

$$K\phi = \omega_n^2 M\phi \tag{2}$$

For the **Euler-Bernoulli beam** with $n$ elements of length $\ell$, the analytical first eigenfrequency (clamped-free) is:

$$\omega_1 = \frac{(1.875)^2}{L^2}\sqrt{\frac{EI}{\rho A}} \tag{3}$$

FEM convergence: eigenfrequency error decreases as $O(h^4)$ for Hermitian beam elements.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

from nlvib.systems.oscillators import SingleMassOscillator, ChainOfOscillators
from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.nonlinearities.elements import cubic_spring

print("Imports OK")

In [ ]:
# SDOF: trivial 1x1 matrices
sdof = SingleMassOscillator(m=1.0, d=0.02, k=1.0)
print("SDOF system:")
print(f"  M = {sdof.M.toarray()}")
print(f"  D = {sdof.D.toarray()}")
print(f"  K = {sdof.K.toarray()}")
print(f"  omega_n = {np.sqrt(sdof.K[0,0]/sdof.M[0,0]):.4f} rad/s")

# Add a cubic spring to make it a Duffing oscillator
sdof.add_nonlinear_element(cubic_spring(k3=0.5, dof_index=0))
print(f"\nAfter adding cubic spring: {len(sdof._nonlinear_elements)} nonlinear element(s)")

In [ ]:
# 4-DOF ChainOfOscillators — show K matrix as heatmap
n_chain = 4        # ← try changing this (number of masses)
m_val   = 1.0      # ← try changing this (mass)
k_val   = 1.0      # ← try changing this (spring stiffness)

chain = ChainOfOscillators(
    masses      = [m_val] * n_chain,
    stiffnesses = [k_val] * (n_chain + 1),  # n+1 springs including boundary
    dampings    = [0.0]   * (n_chain + 1),
)

K_dense = chain.K.toarray()
M_dense = chain.M.toarray()

# Eigenfrequencies
vals, vecs = la.eigh(K_dense, M_dense)
freqs = np.sqrt(np.maximum(vals, 0.0))

print(f"ChainOfOscillators({n_chain} DOF):")
print("K =\n", np.round(K_dense, 2))
print(f"\nNatural frequencies (rad/s): {np.round(freqs, 4)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im = ax1.imshow(K_dense, cmap='RdBu_r', vmin=-np.max(np.abs(K_dense)), vmax=np.max(np.abs(K_dense)))
fig.colorbar(im, ax=ax1)
ax1.set_title(f'K matrix ({n_chain}-DOF chain of oscillators)')
ax1.set_xlabel('DOF index')
ax1.set_ylabel('DOF index')
ax1.set_xticks(range(n_chain))
ax1.set_yticks(range(n_chain))

ax2.bar(range(1, n_chain+1), freqs, color='tab:blue')
ax2.set_xlabel('Mode number')
ax2.set_ylabel('$\\omega_n$ (rad/s)')
ax2.set_title('Natural frequencies')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig

In [ ]:
# FE Euler-Bernoulli beam: eigenfrequency convergence study
# Beam properties (steel-like rod)
L    = 1.0    # length [m]
E    = 2e11   # Young's modulus [Pa]
I    = 1e-5   # second moment of area [m^4]
rho  = 7800.0 # density [kg/m^3]
A    = 1e-3   # cross-section area [m^2]

# Analytical first three eigenfrequencies (clamped-free beam, K&G App. A)
beta_L = np.array([1.8751, 4.6941, 7.8548])  # beta*L for first 3 modes
omega_analytical = (beta_L / L)**2 * np.sqrt(E * I / (rho * A))
print("Analytical eigenfrequencies (rad/s):", np.round(omega_analytical, 2))

n_elem_list = [2, 3, 4, 6, 8, 10, 15]  # ← try adding more elements
errors = np.zeros((len(n_elem_list), 3))  # errors for modes 1-3

for idx, n_el in enumerate(n_elem_list):
    beam = FE_EulerBernoulliBeam(
        n_elements=n_el, L=L, E=E, I_area=I, rho=rho, A=A, bc='clamped-free'
    )
    K_b = beam.K.toarray()
    M_b = beam.M.toarray()
    vals_b, _ = la.eigh(K_b, M_b)
    freqs_b = np.sqrt(np.maximum(vals_b, 0.0))
    # Relative error for first 3 modes
    for m in range(3):
        if m < len(freqs_b):
            errors[idx, m] = abs(freqs_b[m] - omega_analytical[m]) / omega_analytical[m]

fig, ax = plt.subplots(figsize=(8, 4))
for m in range(3):
    ax.semilogy(n_elem_list, errors[:, m], 'o-', label=f'Mode {m+1}')
ax.set_xlabel('Number of elements $n$')
ax.set_ylabel('Relative eigenfrequency error')
ax.set_title('FE Euler-Bernoulli beam — eigenfrequency convergence (clamped-free)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig

## Try it yourself

- Change `n_chain = 4` to `n_chain = 8` — observe how the tridiagonal K matrix grows.
- Change `n_elem_list` to include `[20, 30]` — watch mode-3 error approach machine precision.
- Set `bc='clamped-clamped'` in `FE_EulerBernoulliBeam` — boundary conditions change the analytical solution.

## Key Takeaways

- `SingleMassOscillator` and `ChainOfOscillators` give 1-DOF to n-DOF lumped models.
- `FE_EulerBernoulliBeam` assembles global M/K matrices from element matrices — fully sparse.
- FEM eigenfrequency error scales as $O(1/n^4)$ for Hermitian beam elements — 10 elements gives < 0.01% error for mode 1.